[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P02/P02-1_Calculer_Parcourir_CORRIGE.ipynb)

# Calculs, listes et boucles

Chapitre 2 du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

**Contexte industriel.** Votre fournisseur annonce ses limites en psi, votre plan les donne en mégapascals, et votre banc de traction livre ses résultats par lot ?

**Objectif terminal.** Ce chapitre montre comment calculer sans erreur de conversion, juger une approximation, puis ranger, parcourir et trier des mesures, avec Python employé comme une calculatrice qui garde la mémoire de ses calculs.

**Objectif instrumental.**

- Écrire un calcul d'ingénieur avec des variables qui portent leur unité.
- Présenter un résultat, convertir des unités, juger une approximation.
- Ranger, parcourir et trier des mesures : liste, dictionnaire, `for`, `if`.

**Critère de résultat.** Chaque exercice atteint son résultat attendu, que des contrôles du notebook vérifient.

**Vérification.** Beaucoup d'exercices donnent une seconde voie : un calcul fait par deux chemins indépendants est un calcul qu'on peut croire.

**Prérequis.** Aucun après une filière scientifique.

## Source de données

Aucun jeu de données : les valeurs viennent de l'énoncé ou sont engendrées par le code, à graine fixée.

## Exercices

| N° | Exercice | Durée indicative |
|:-|:-|-:|
| 2.1 | Contrainte dans une tige | 10 min |
| 2.2 | Des mégapascals aux psi | 10 min |
| 2.3 | L'approximation 355/113 de $\pi$ | 15 min |
| 2.4 | Moyenne de cinq essais de traction | 10 min |
| 2.5 | Fraction continue de $\pi$ | 10 min |
| 2.6 | Les réduites de $\pi$ (défi) | 15 min |

Dans le sujet, les cellules marquées `# Votre code :` sont à écrire ; le corrigé les donne résolues. Chaque exercice se termine par des contrôles qui comparent votre résultat à celui du livre.

### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [1]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())

Python 3.12.0


### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook n'a besoin d'aucun jeu ni d'aucun module : la cellule ne signale rien, mais elle prépare l'import des modules du livre dans vos propres cellules.

In [2]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = []
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Préparation

In [3]:
import math
from fractions import Fraction

## Exercice 2.1 : Contrainte dans une tige (10 min)

Une tige de traction a un diamètre de 10 mm. Elle porte un effort de 12 kN. Calculez sa section en mm², puis la contrainte normale en mégapascals (un newton par mm² vaut un mégapascal), avec des variables qui portent leur unité.

**Résultat attendu.** Section de 78,54 mm², contrainte de 152,79 MPa.

**Seconde voie.** À la calculatrice, la section vaut $25\,\pi$ mm² ; l'effort divisé par ce nombre redonne 152,79 MPa.

**Indication.** La section d'un disque vaut $\pi d^2/4$. La contrainte est l'effort en newtons divisé par la section en mm², c'est-à-dire des mégapascals. Pensez à convertir les kilonewtons en newtons.

Les contrôles de la cellule lisent la variable `sigma_MPa` : gardez ce nom.

In [4]:
F_N = 12000.0                        # effort de traction, en newtons
d_mm = 10.0                          # diamètre, en millimètres
# Solution :
aire_mm2 = math.pi * d_mm**2 / 4     # section, en mm²
sigma_MPa = F_N / aire_mm2           # des N/mm², c'est-à-dire des MPa
print(f"section : {aire_mm2:.2f} mm²")
print(f"contrainte : {sigma_MPa:.2f} MPa")
# Fin de la solution
assert abs(sigma_MPa - 152.79) < 0.005, "contrainte attendue : 152,79 MPa"
print("Exercice 2.1 : contrôles passés")

section : 78.54 mm²
contrainte : 152.79 MPa
Exercice 2.1 : contrôles passés


## Exercice 2.2 : Des mégapascals aux psi (10 min)

Un fournisseur américain donne ses limites en psi, des livres-force par pouce carré. À partir des définitions de la section, retrouvez la valeur d'un mégapascal en psi, puis convertissez la contrainte de l'exercice 2.1. Affichez les résultats avec une f-string.

**Résultat attendu.** Un psi vaut 0,006895 MPa, soit 145,04 psi par mégapascal ; la tige de l'exercice 2.1 supporte 22 160 psi.

**Seconde voie.** Une table des unités donne un psi à environ $6{,}9$ kPa : le facteur obtenu en mégapascals doit s'accorder avec ce repère.

**Indication.** Une livre-force vaut $0{,}45359237 \times 9{,}80665$ newtons, masse en kilogrammes fois pesanteur. Divisez par le carré de $25{,}4$ mm pour obtenir un psi en mégapascals, puis prenez l'inverse. Multipliez la contrainte en mégapascals par ce facteur pour la lire en psi.

Les contrôles de la cellule lisent la variable `psi_par_MPa` : gardez ce nom.

In [5]:
livre_force_N = 0.45359237 * 9.80665     # 1 lbf : le poids d'une masse de 0,45359237 kg
pouce_mm = 25.4                          # 1 pouce, en millimètres
# Solution :
psi_en_MPa = livre_force_N / pouce_mm**2 # 1 psi = 1 lbf par pouce carré
psi_par_MPa = 1 / psi_en_MPa
print(f"1 psi = {psi_en_MPa:.6f} MPa")
print(f"1 MPa = {psi_par_MPa:.2f} psi")
print(f"la tige de l'exercice 2.1 : {sigma_MPa * psi_par_MPa:.0f} psi")
# Fin de la solution
assert abs(psi_par_MPa - 145.04) < 0.005, "attendu : 145,04 psi par MPa"
print("Exercice 2.2 : contrôles passés")

1 psi = 0.006895 MPa
1 MPa = 145.04 psi
la tige de l'exercice 2.1 : 22160 psi
Exercice 2.2 : contrôles passés


## Exercice 2.3 : L'approximation 355/113 de $\pi$ (15 min)

Calculez l'erreur absolue et l'erreur relative de $355/113$, comparez-la à l'erreur de $22/7$, puis demandez au module `fractions` la meilleure fraction de dénominateur au plus $1000$.

**Résultat attendu.** Erreur absolue de ${2{,}67 \times 10^{-7}}$, erreur relative de ${8{,}49 \times 10^{-8}}$ ; pour $22/7$, l'erreur absolue est de ${1{,}26 \times 10^{-3}}$ ; meilleure fraction : 355/113.

**Seconde voie.** Soustrayez à la main 3,1415929204 ($355/113$) et 3,1415926536 ($\pi$) : l'écart doit s'accorder avec l'erreur absolue.

**Indication.** L'erreur absolue est la différence entre l'approximation et `math.pi` ; l'erreur relative la divise par $\pi$. La classe `Fraction` reçoit `math.pi`, et sa méthode `limit_denominator(1000)` rend la fraction cherchée. Le notebook `P02-1` du dépôt la détaille.

Les contrôles de la cellule lisent les variables `erreur`, `erreur_relative`, `meilleure` : gardez ces noms.

In [6]:
approche = 355 / 113
# Solution :
erreur = approche - math.pi
erreur_relative = erreur / math.pi
print(f"355/113 = {approche:.10f}")
print(f"pi      = {math.pi:.10f}")
print(f"erreur absolue  : {erreur:.2e}")
print(f"erreur relative : {erreur_relative:.2e}")
print(f"pour comparer, 22/7 - pi = {22 / 7 - math.pi:.2e}")
meilleure = Fraction(math.pi).limit_denominator(1000)
print("meilleure fraction de dénominateur au plus 1000 :", meilleure)
# Fin de la solution
assert abs(erreur - 2.67e-7) < 0.005e-7, "erreur absolue attendue : 2,67e-7"
assert abs(erreur_relative - 8.49e-8) < 0.005e-8, "erreur relative attendue : 8,49e-8"
assert meilleure == Fraction(355, 113), "meilleure fraction attendue : 355/113"
print("Exercice 2.3 : contrôles passés")

355/113 = 3.1415929204
pi      = 3.1415926536
erreur absolue  : 2.67e-07
erreur relative : 8.49e-08
pour comparer, 22/7 - pi = 1.26e-03
meilleure fraction de dénominateur au plus 1000 : 355/113
Exercice 2.3 : contrôles passés


## Exercice 2.4 : Moyenne de cinq essais de traction (10 min)

Cinq éprouvettes d'un même lot ont donné ces résistances à la traction, en MPa : $412{,}0$ ; $428{,}5$ ; $405{,}2$ ; $441{,}8$ ; $419{,}3$. Rangez-les dans une liste, affichez le premier et le dernier, les trois premiers par une tranche, puis le plus faible, le plus fort et la moyenne. Rangez enfin le repère du lot, `"L-17"`, sa nuance, `"S355"`, et ses essais dans un dictionnaire à trois clés.

**Résultat attendu.** 5 essais ; plus faible de 405,2 MPa, plus fort de 441,8 MPa, moyenne de 421,4 MPa.

**Seconde voie.** Une fois la section suivante lue, refaites la somme sans `sum`, avec une variable de cumul que la boucle fait croître ; divisée par 5, elle redonne 421,4 MPa.

**Indication.** La moyenne est `sum(liste) / len(liste)`. Le premier essai est `liste[0]`, le dernier `liste[-1]`, les trois premiers `liste[0:3]`. Le dictionnaire associe un nom à chaque valeur, par exemple `"essais_MPa"` à la liste.

Les contrôles de la cellule lisent la variable `moyenne_MPa` : gardez ce nom.

In [7]:
essais_MPa = [412.0, 428.5, 405.2, 441.8, 419.3]
# Solution :
n = len(essais_MPa)
moyenne_MPa = sum(essais_MPa) / n
print("nombre d'essais :", n)
print("premier :", essais_MPa[0], " dernier :", essais_MPa[-1])
print("trois premiers :", essais_MPa[0:3])
print("plus faible :", min(essais_MPa), " plus fort :", max(essais_MPa))
print("moyenne :", round(moyenne_MPa, 1), "MPa")
sous_moyenne = 0

for i, s in enumerate(essais_MPa):
    if s < moyenne_MPa:
        print(f"essai n° {i + 1} : {s} MPa, sous la moyenne")
        sous_moyenne = sous_moyenne + 1

print("essais sous la moyenne :", sous_moyenne)
lot = {"repere": "L-17", "nuance": "S355", "essais_MPa": essais_MPa}
print("clés de la fiche du lot :", list(lot.keys()))
# Fin de la solution
assert abs(moyenne_MPa - 421.4) < 0.05, "moyenne attendue : 421,4 MPa"
print("Exercice 2.4 : contrôles passés")

nombre d'essais : 5
premier : 412.0  dernier : 419.3
trois premiers : [412.0, 428.5, 405.2]
plus faible : 405.2  plus fort : 441.8
moyenne : 421.4 MPa
essai n° 1 : 412.0 MPa, sous la moyenne
essai n° 3 : 405.2 MPa, sous la moyenne
essai n° 5 : 419.3 MPa, sous la moyenne
essais sous la moyenne : 3
clés de la fiche du lot : ['repere', 'nuance', 'essais_MPa']
Exercice 2.4 : contrôles passés


## Exercice 2.5 : Fraction continue de $\pi$ (10 min)

Une fraction continue écrit un nombre comme une partie entière, plus l'inverse d'un reste lui-même écrit de la même façon. Pour $\pi$, gardez la partie entière, inversez le reste, recommencez, et conservez les cinq premiers termes dans une liste.

**Résultat attendu.** `[3, 7, 15, 1, 292]`.

**Seconde voie.** La même boucle, appliquée à la fraction exacte 355/113 de la section 2.3 avec `Fraction`, atteint un reste nul après trois termes dont les deux premiers sont ceux de $\pi$ ; un test sur le reste, avant de l'inverser, sort alors de la boucle, avec le mot `break` de la section suivante.

**Indication.** Dans une boucle `for k in range(5)`, la partie entière s'obtient par `a = int(x)`, le reste par `x - a`, et le nouveau `x` est l'inverse de ce reste. Ajoutez chaque partie entière à la liste des termes.

Les contrôles de la cellule lisent la variable `termes` : gardez ce nom.

In [8]:
# Solution :
x = math.pi
termes = []

for k in range(5):
    a = int(x)            # partie entière
    termes.append(a)
    x = 1 / (x - a)       # inverse du reste

print("fraction continue de pi :", termes)
# Fin de la solution
assert termes == [3, 7, 15, 1, 292], "termes attendus : [3, 7, 15, 1, 292]"
print("Exercice 2.5 : contrôles passés")

fraction continue de pi : [3, 7, 15, 1, 292]
Exercice 2.5 : contrôles passés


## Exercice 2.6 : Les réduites de $\pi$ (défi, 15 min)

Les réduites $p/q$ de la fraction continue se calculent de proche en proche : $p_k = a_k\,p_{k-1} + p_{k-2}$ et $q_k = a_k\,q_{k-1} + q_{k-2}$ pour $k \geq 1$, avec $p_{-1} = 1$, $q_{-1} = 0$, $p_0 = a_0$ et $q_0 = 1$. Calculez les cinq premières réduites et leur écart algébrique $p/q - \pi$. Quelle réduite suit $355/113$, et quelle est son erreur ?

**Résultat attendu.** La réduite qui suit $355/113$ est $103\,993/33\,102$, avec une erreur de ${-5{,}78 \times 10^{-10}}$ ; celle de $355/113$ vaut ${2{,}67 \times 10^{-7}}$.

**Seconde voie.** `Fraction(math.pi).limit_denominator(33102)`, avec le plus grand dénominateur de la liste, rend la meilleure fraction à cette borne : la réduite $103\,993/33\,102$.

**Indication.** Gardez en mémoire les deux dernières valeurs de $p$ et de $q$. À chaque tour de boucle, calculez les nouvelles valeurs avec les formules, puis décalez. L'erreur est `p / q - math.pi`. Le notebook `P02-1` du dépôt détaille la boucle.

Les contrôles de la cellule lisent les variables `erreur5`, `p5`, `q5` : gardez ces noms.

In [9]:
# Solution :
p_avant, p = 1, termes[0]
q_avant, q = 0, 1
reduites = [(p, q)]

for a in termes[1:]:
    p_avant, p = p, a * p + p_avant
    q_avant, q = q, a * q + q_avant
    reduites.append((p, q))

for p, q in reduites:
    print(f"{p:>6}/{q:<6} erreur : {p / q - math.pi:+.2e}")

p5, q5 = reduites[4]
erreur5 = p5 / q5 - math.pi
print("la réduite qui suit 355/113 :", p5, "/", q5)
# Fin de la solution
assert (p5, q5) == (103993, 33102), "réduite attendue : 103993/33102"
assert abs(erreur5 - (-5.78e-10)) < 0.005e-10, "erreur attendue : -5,78e-10"
print("Exercice 2.6 : contrôles passés")

     3/1      erreur : -1.42e-01
    22/7      erreur : +1.26e-03
   333/106    erreur : -8.32e-05
   355/113    erreur : +2.67e-07
103993/33102  erreur : -5.78e-10
la réduite qui suit 355/113 : 103993 / 33102
Exercice 2.6 : contrôles passés


## Essais des fiches de fonction du chapitre

Chaque fiche du livre donne l'appel minimal d'une fonction. Les cellules suivantes rejouent ces appels, avec les mêmes lignes de code et la même sortie que la fiche : changez une valeur et relisez le résultat.

### Fiche F02-1 : Le module math (`math`)

In [10]:
import math

F_kN = 15.0                               # effort sur l'ancrage, kN
angle_deg = 40.0                          # inclinaison sur l'axe, degrés
angle_rad = math.radians(angle_deg)       # angle converti, radians
N_kN = F_kN * math.cos(angle_rad)         # effort axial de traction, kN
T_kN = F_kN * math.sin(angle_rad)         # effort transversal, kN
norme_kN = math.hypot(N_kN, T_kN)         # norme du vecteur résultant, kN

print(f"N = {N_kN:.2f} kN, T = {T_kN:.2f} kN".replace(".", ","))
print(math.isclose(norme_kN, F_kN))       # équilibre reconstitué

N = 11,49 kN, T = 9,64 kN
True
